# 02 · 关联测试与 FWL 残差化 Association Testing & Frisch-Waugh-Lovell

> **能力标签**：GB2（关联测试与表型 / Association Testing & Phenotype）

## 目标 Objectives

完成本课后，你应该能够：

1. 用线性回归对单个 SNP 进行关联测试，计算 $\hat{\beta}$（效应量 effect size）、**SE**（标准误 standard error）、$t$ 统计量、$p$ 值。
2. 解释 **协变量（covariate）** 在 GWAS 中的作用——控制混杂因子，如性别、年龄、群体主成分。
3. 陈述并应用 **Frisch-Waugh-Lovell (FWL) 定理**：先残差化再回归，等价于联合回归。
4. 编写**向量化（vectorized）**实现，一次性对 $m$ 个 SNP 并行计算，避免逐列 for 循环。
5. 绘制 **Manhattan 图**，直观显示全基因组 $-\log_{10}(p)$ 分布。

> 对应能力：**GB2**


## 概念讲解 Concepts

### 1 · 单 SNP 线性回归 Single-SNP Linear Regression

对每个 SNP $j$，将其剂量 $g_j \in \mathbb{R}^n$ 作为自变量，表型 $y \in \mathbb{R}^n$ 为因变量：

$$y = \mu + \beta_j g_j + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I)$$

OLS 估计量（ordinary least squares estimate）：

$$\hat{\beta}_j = \frac{\sum_i (g_{ij} - \bar{g}_j)(y_i - \bar{y})}{\sum_i (g_{ij} - \bar{g}_j)^2}$$

残差方差（residual variance）：

$$\hat{\sigma}^2 = \frac{\|y - \hat{y}\|^2}{n - 2}$$

标准误（standard error）：

$$\text{SE}(\hat{\beta}_j) = \sqrt{\frac{\hat{\sigma}^2}{\sum_i (g_{ij} - \bar{g}_j)^2}}$$

$t$ 统计量：

$$t_j = \frac{\hat{\beta}_j}{\text{SE}(\hat{\beta}_j)}$$

双侧 $p$ 值：$p_j = 2 \cdot P(T_{n-2} > |t_j|)$，其中 $T_{n-2}$ 为自由度 $n-2$ 的 $t$ 分布。

---

### 2 · 协变量与混杂 Covariates & Confounding

实际 GWAS 中，表型受多种非遗传因素影响：性别（sex）、年龄（age）、
群体分层（population stratification，用主成分 PC 表示）。
直接不加控制地回归会产生虚假关联（spurious association）。

解决方案：**联合回归（joint regression）**，把协变量矩阵 $C \in \mathbb{R}^{n \times k}$（含截距列）与 SNP 一起放入模型。

---

### 3 · Frisch-Waugh-Lovell 定理 FWL Theorem

**定理**：在含协变量 $C$ 的联合回归 $y \sim g_j + C$ 中，$g_j$ 的 OLS 系数 $\hat{\beta}_j$ 等于：
先把 $y$ 和 $g_j$ 分别在 $C$ 上残差化（project out $C$ 的列空间），
再对残差做简单回归所得系数。

$$\tilde{y} = y - C(C^\top C)^{-1} C^\top y, \quad \tilde{g}_j = g_j - C(C^\top C)^{-1} C^\top g_j$$

$$\hat{\beta}_j = \frac{\tilde{g}_j^\top \tilde{y}}{\tilde{g}_j^\top \tilde{g}_j}$$

**优势**：协变量矩阵 $C$ 只需分解一次（$O(nk^2)$），
然后 $m$ 个 SNP 全部用向量化矩阵乘法处理（$O(nm)$），大幅提速。

---

### 4 · 向量化实现 Vectorized Implementation

利用 FWL，整个 $m$ 个 SNP 的计算可以向量化：

```python
Cpinv = pinv(C)                          # (k, n)
yr    = y - C @ (Cpinv @ y)              # 表型残差 (n,)
Gr    = G - C @ (Cpinv @ G)              # 基因型残差矩阵 (n, m)
Sxx   = (Gr**2).sum(axis=0)              # 每个 SNP 的 sum-of-squares (m,)
beta  = (Gr * yr[:,None]).sum(axis=0) / Sxx
```

一次矩阵运算即可得到所有 $m$ 个 $\hat{\beta}$。

---

### 5 · Manhattan 图 Manhattan Plot

Manhattan 图：x 轴为染色体（chromosome）位置，y 轴为 $-\log_{10}(p)$。
通常画两条参考线：
- **名义显著水平（nominal significance）**：$p < 10^{-5}$，即 $y = 5$
- **全基因组显著（genome-wide significance）**：$p < 5 \times 10^{-8}$，即 $y \approx 7.3$

这个阈值来源于：约 $10^6$ 个独立 SNP 的 Bonferroni 校正（$0.05 / 10^6 = 5 \times 10^{-8}$）。


## 示例 Worked Example

合成 GWAS 数据演示线性回归、FWL 残差化、对照 statsmodels，并绘制 Manhattan 图。

In [ ]:
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile, os

rng = np.random.default_rng(123)

# 合成数据
n, m = 300, 2000
k_cov = 3

true_aaf = rng.uniform(0.1, 0.9, m)
G = rng.binomial(2, true_aaf, (n, m)).astype(float)
covariates = rng.standard_normal((n, k_cov))

true_beta = np.zeros(m)
true_beta[0] = 0.4
true_beta[1] = -0.3
cov_effects = covariates @ rng.standard_normal(k_cov)
y = G @ true_beta + cov_effects + rng.standard_normal(n)

print(f"G.shape = {G.shape},  y.shape = {y.shape}")
print(f"协变量矩阵 shape = {covariates.shape}")


In [ ]:
# 实现 assoc：FWL 残差化向量化关联测试
def assoc(G, y, covariates=None):
    """对每个 SNP 做单变量线性回归；用 FWL 处理协变量。
    G:(n,m) 基因型; y:(n,); covariates:(n,k) 或 None（截距总是包含）。
    返回 dict 包含 beta, se, t, p，每个长度 m。"""
    G = np.asarray(G, dtype=float)
    y = np.asarray(y, dtype=float)
    n = G.shape[0]
    C = np.ones((n, 1)) if covariates is None else np.column_stack([np.ones(n), covariates])
    Cpinv = np.linalg.pinv(C)

    def resid(M):
        """把矩阵/向量 M 对 C 做残差化。"""
        return M - C @ (Cpinv @ M)

    yr = resid(y)
    Gr = resid(G)
    dof = n - C.shape[1] - 1

    Sxx = (Gr ** 2).sum(axis=0)
    beta = (Gr * yr[:, None]).sum(axis=0) / Sxx
    resid_y = yr[:, None] - Gr * beta[None, :]
    sigma2 = (resid_y ** 2).sum(axis=0) / dof
    se = np.sqrt(sigma2 / Sxx)
    t = beta / se
    p = 2 * stats.t.sf(np.abs(t), dof)
    return {"beta": beta, "se": se, "t": t, "p": p}

results = assoc(G, y, covariates)
print("关联测试完成。结果 keys:", list(results.keys()))
print(f"beta.shape = {results['beta'].shape}")
print(f"真实 causal SNP 0：beta={results['beta'][0]:.4f}  p={results['p'][0]:.2e}")
print(f"真实 causal SNP 1：beta={results['beta'][1]:.4f}  p={results['p'][1]:.2e}")
print(f"零假设 SNP 100：   beta={results['beta'][100]:.4f}  p={results['p'][100]:.4f}")


In [ ]:
# 与 statsmodels OLS 对照（验证单个 SNP）
import statsmodels.api as sm

j = 0
X_sm = sm.add_constant(np.column_stack([G[:, j], covariates]))
ols_fit = sm.OLS(y, X_sm).fit()

print("=== statsmodels OLS (SNP 0) ===")
print(f"beta = {ols_fit.params[1]:.6f}  (assoc: {results['beta'][0]:.6f})")
print(f"SE   = {ols_fit.bse[1]:.6f}  (assoc: {results['se'][0]:.6f})")
print(f"t    = {ols_fit.tvalues[1]:.6f}  (assoc: {results['t'][0]:.6f})")
print(f"p    = {ols_fit.pvalues[1]:.6e}  (assoc: {results['p'][0]:.6e})")
print()
print("最大绝对差异：")
print(f"  |delta_beta| = {abs(ols_fit.params[1] - results['beta'][0]):.2e}")
print(f"  |delta_SE|   = {abs(ols_fit.bse[1]   - results['se'][0]):.2e}")
print(f"  |delta_p|    = {abs(ols_fit.pvalues[1] - results['p'][0]):.2e}")


In [ ]:
# Manhattan 图
tmpdir = tempfile.mkdtemp()

chrom_sizes_raw = [500,400,380,320,300,280,270,260,250,240,
                   220,210,200,190,180,170,160,150,140,130,120,100]
total_raw = sum(chrom_sizes_raw)
chrom_sizes = [max(1, int(c * m / total_raw)) for c in chrom_sizes_raw]
chrom_sizes[-1] += m - sum(chrom_sizes)

chroms = np.concatenate([np.full(sz, c+1) for c, sz in enumerate(chrom_sizes)])
logp = -np.log10(np.clip(results["p"], 1e-300, 1))

palette = ["#2166AC", "#D6604D"]
c_arr = np.where(chroms % 2 == 0, 0, 1)

fig, ax = plt.subplots(figsize=(12, 4))
for c_idx, color in enumerate(palette):
    mask = c_arr == c_idx
    ax.scatter(np.where(mask)[0], logp[mask], c=color, s=2, alpha=0.7, rasterized=True)

ax.axhline(-np.log10(5e-8), color="red", linestyle="--", linewidth=0.8,
           label="全基因组显著 5e-8")
ax.axhline(-np.log10(1e-5), color="gray", linestyle=":", linewidth=0.8,
           label="名义显著 1e-5")
for cj in [0, 1]:
    ax.annotate(f"SNP {cj}", xy=(cj, logp[cj]),
                xytext=(cj+30, logp[cj]+0.5),
                fontsize=7, color="black",
                arrowprops=dict(arrowstyle="->", lw=0.8))

ax.set_xlabel("SNP index (按染色体排列)")
ax.set_ylabel("-log10(p)")
ax.set_title("Manhattan 图 Manhattan Plot (合成数据)")
ax.legend(fontsize=8)
fig.tight_layout()

out = os.path.join(tmpdir, "manhattan.png")
fig.savefig(out, dpi=120)
plt.close(fig)
print(f"Manhattan 图已保存：{out}")
print(f"超过全基因组显著阈值的 SNP 数：{(results['p'] < 5e-8).sum()}")


In [ ]:
# FWL 定理数值验证
print("=== FWL 定理数值验证 ===")

C = np.column_stack([np.ones(n), covariates])
Cpinv = np.linalg.pinv(C)

# 方法 A：FWL 残差化后简单回归
yr = y - C @ (Cpinv @ y)
gr0 = G[:, 0] - C @ (Cpinv @ G[:, 0])
beta_fwl = (gr0 @ yr) / (gr0 @ gr0)

# 方法 B：联合回归（矩阵求解）
X_joint = np.column_stack([G[:, 0], covariates, np.ones(n)])
coeffs_joint, _, _, _ = np.linalg.lstsq(X_joint, y, rcond=None)
beta_joint = coeffs_joint[0]

print(f"方法 A（FWL 残差化）：beta = {beta_fwl:.10f}")
print(f"方法 B（联合回归）：  beta = {beta_joint:.10f}")
print(f"|差异| = {abs(beta_fwl - beta_joint):.2e}  (数值误差，近似 0)")
print("FWL 定理验证通过")


## 动手 Exercise

以下合成数据集 `G_ex`、`y_ex`、`cov_ex` 已准备好。请你：

1. **不调用** `assoc`，手动用 NumPy 实现 FWL 残差化，计算 SNP 3 的 $\hat{\beta}$、SE、$t$、$p$。
2. 调用 `assoc(G_ex, y_ex, cov_ex)` 验证你的结果与函数输出一致（允许数值误差 $< 10^{-10}$）。
3. 在 `results_ex` 中找出 $p < 0.05$ 的 SNP 个数，并打印其索引。

```python
rng_ex = np.random.default_rng(99)
n_ex, m_ex = 150, 100
G_ex = rng_ex.binomial(2, 0.3, (n_ex, m_ex)).astype(float)
cov_ex = rng_ex.standard_normal((n_ex, 2))
y_ex = G_ex[:, 5] * 0.5 + cov_ex[:, 0] * 0.3 + rng_ex.standard_normal(n_ex)

# 你的代码写在这里
```

> **提示**：自由度 dof = n - (k+1) - 1，其中 k 是协变量数（不含截距列）。


## 延伸阅读 Further Reading

1. **Frisch & Waugh (1933)**. "Partial time regressions as compared with individual trends." *Econometrica.* FWL 定理的原始论文。
2. **Yang et al. (2014)**. "Conditional and joint multiple-SNP analysis of GWAS summary statistics." *Nature Genetics.*
3. **Price et al. (2006)**. "Principal components analysis corrects for stratification in genome-wide association studies." *Nature Genetics.*
4. `statsmodels` 文档 `sm.OLS`：用于验证与对照实现。


## 关联练习 Related Assignment

完成 **b7-assoc** 编程作业：在 `progress/<你>/work/b7-assoc/solution.py` 中实现
`assoc(G, y, covariates=None)` 函数，返回包含 `beta`、`se`、`t`、`p` 的字典。

评测命令：

```bash
python check.py b7-assoc
```

> 本课的 `assoc` 实现与作业签名完全一致，可直接参照上方代码编写。
